In [164]:
import os
from torch.utils.data import Dataset, DataLoader
from typing import List, Tuple

In [165]:
class LargeFileDataset(Dataset[Tuple[str, str]]):
    def __init__(self, root_dir: str, extensions: List[str]):
        self.file_paths: List[str] = []
        self.extensions = tuple(extensions)
        for dirpath, _, filenames in os.walk(root_dir):
            for fname in filenames:
                if fname.endswith(self.extensions):
                    self.file_paths.append(os.path.join(dirpath, fname))

    def __len__(self) -> int:
        return len(self.file_paths)

    def __getitem__(self, idx: int) -> Tuple[str, str]:
        file_path = self.file_paths[idx]
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        return content, file_path

In [ ]:
supported_extensions = [
    '.md', '.html', '.css',
    '.js', '.ts', '.jsx', '.tsx',
    '.py',
    '.json', '.yaml', '.yml',
]

dataset = LargeFileDataset("<dataset directory>", extensions=supported_extensions)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

print(f"Total files found: {len(dataset)}")

Total files found: 64


In [167]:
def print_file_lengths_summary(dataset: LargeFileDataset):
    lengths: List[int] = []
    for idx in range(len(dataset)):
        try:
            content, _ = dataset[idx]
            lengths.append(len(content))
        except Exception as e:
            print(f"Error reading sample {idx}: {e}")

    if lengths:
        min_len = min(lengths)
        max_len = max(lengths)
        mean_len = sum(lengths) // len(lengths)

        print(f"File count : {len(lengths):>15,}")
        print(f"Min length : {min_len:>15,}")
        print(f"Max length : {max_len:>15,}")
        print(f"Mean length: {mean_len:>15,}")
    else:
        print("No files found or all failed to read.")

# Execute to see the summary
print_file_lengths_summary(dataset)

File count :              64
Min length :              92
Max length :         162,396
Mean length:           6,760


# Tokenizer

In [168]:
# %pip install torchinfo

In [169]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torchinfo import summary as torchinfo_summary

In [170]:
def stringToTensor(s: str) -> torch.Tensor:
    return torch.tensor([ord(c) for c in s], dtype=torch.long)

def stringBatchToTensor(batch: List[str]) -> torch.Tensor:
    max_len = max(len(s) for s in batch)
    tensor = torch.zeros((len(batch), max_len), dtype=torch.long)
    for i, s in enumerate(batch):
        tensor[i, :len(s)] = stringToTensor(s)
    return tensor

In [171]:
class TokenizerModel(nn.Module):
    def __init__(
            self,
            max_sequence_length: int = 10_000_000,
            subword_count_limit: int = 100_000,
            window_sizes: List[int] = [3, 5, 7, 9]
        ):
        super().__init__()
        self.max_sequence_length = max_sequence_length
        self.subword_count_limit = subword_count_limit

        self.window_sizes = window_sizes
        self.num_kernels = len(window_sizes)
        # ----- Latent layers -----

        self.kernels = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(in_channels=1, out_channels=1, kernel_size=ws, padding=ws//2),
                nn.SiLU(),
            )
            for ws in window_sizes
        ])

        self.kernel_modifiers = nn.Parameter(
            torch.ones(self.num_kernels) / self.num_kernels
        )

        self.augment = nn.Sequential(
            nn.Conv1d(
                in_channels=self.num_kernels,
                out_channels=self.num_kernels,
                kernel_size=3, padding=1
            ),
            nn.GroupNorm(num_groups=1, num_channels=self.num_kernels),
            nn.SiLU()
        )

        self.splice_points = nn.Sequential(
            nn.Conv1d(
                in_channels=self.num_kernels,
                out_channels=1,
                kernel_size=3, padding=1, bias=False
            ),
            nn.SiLU(),
            nn.Conv1d(
                in_channels=1,
                out_channels=1, kernel_size=3, padding=1
            ),
            nn.Sigmoid()
        )

    def forward(self, seq: torch.Tensor) -> torch.Tensor:
        x = seq.float() # Shape: (batch_size, seq_length)
        x.unsqueeze_(1) # Shape: (batch_size, 1, seq_length)

        # Truncate to max sequence length
        if x.size(2) > self.max_sequence_length:
            x = x[:, :, :self.max_sequence_length]
            print(f"[WARN] Input sequence length {x.size(2)} exceeds max of {self.max_sequence_length}.")

        conv_outputs = [
            kernel(x) * self.kernel_modifiers[i]
            for i, kernel in enumerate(self.kernels)
        ]
        x = torch.cat(conv_outputs, dim=1)  # Shape: (batch_size, num_kernels, seq_length)
        x = self.augment(x)                 # Shape: (batch_size, num_kernels, seq_length)
        x = self.splice_points(x)           # Shape: (batch_size, 1, seq_length)

        x = x.squeeze(1)  # Remove channel dimension
        return 2 * x - 1  # Scale to [-1, 1]

In [172]:
tokenizer = TokenizerModel()

In [173]:
torchinfo_summary(
    tokenizer,
    input_data=stringBatchToTensor(next(iter(dataloader))[0])
)

Layer (type:depth-idx)                   Output Shape              Param #
TokenizerModel                           [4, 16827]                4
├─ModuleList: 1-1                        --                        --
│    └─Sequential: 2-1                   [4, 1, 16827]             --
│    │    └─Conv1d: 3-1                  [4, 1, 16827]             4
│    │    └─SiLU: 3-2                    [4, 1, 16827]             --
│    └─Sequential: 2-2                   [4, 1, 16827]             --
│    │    └─Conv1d: 3-3                  [4, 1, 16827]             6
│    │    └─SiLU: 3-4                    [4, 1, 16827]             --
│    └─Sequential: 2-3                   [4, 1, 16827]             --
│    │    └─Conv1d: 3-5                  [4, 1, 16827]             8
│    │    └─SiLU: 3-6                    [4, 1, 16827]             --
│    └─Sequential: 2-4                   [4, 1, 16827]             --
│    │    └─Conv1d: 3-7                  [4, 1, 16827]             10
│    │    └─SiLU: 3

### Tokenizer - Loss Function

In [174]:
from typing import TypedDict

In [175]:
class TokenizerLossStats(TypedDict):
    l_budget: float
    l_gap: float
    l_efficiency: float
    l_sharp: float
    l_contrast: float

In [176]:
class TokenizerLoss():
    def __init__(self, model: TokenizerModel):
        super().__init__()
        self.subword_count_limit = model.subword_count_limit
        self.max_gap = max(model.window_sizes)

    def __call__(self, y_pred: torch.Tensor) -> Tuple[torch.Tensor, TokenizerLossStats]:
        # y_pred shape: (batch_size, seq_length), values in [-1, 1]
        sequence_length = y_pred.size(1)

        activations = torch.sigmoid(8 * y_pred - 4) # Shape: (batch_size, seq_length)
        total_activations = activations.sum(dim=1)  # Shape: (batch_size,)

        # Sparsity Loss (Enforce subword count limit)
        l_budget = F.relu(total_activations - self.subword_count_limit).mean()

        # Maximum Gap Loss (Force a partition at least every 9-10 chars)
        y_maxpooled = F.max_pool1d(
            y_pred.unsqueeze(1),        # Shape: (batch_size, 1, seq_length)
            kernel_size=self.max_gap,
            stride=1,
            padding=self.max_gap // 2
        )                               # Shape: (batch_size, 1, seq_length)
        l_gap = F.relu(1 - y_maxpooled).mean()

        # Minimization Objective (Efficiency)
        mean_density = total_activations.mean() / sequence_length
        target_density = torch.tensor(1.0 / self.max_gap, device=mean_density.device)
        l_efficiency = F.mse_loss(mean_density, target_density)

        # Bipolar Sharpness (Push toward -1 or 1, avoid 0)
        l_sharp = (1.0 - torch.square(y_pred)).mean()

        # Contrast Loss: Penalize boundaries that are wide
        local_diff = torch.abs(0.2 - y_pred[:, 1:] + y_pred[:, :-1])
        contrast_weight = torch.abs(y_pred[:, 1:])
        l_contrast = (local_diff * contrast_weight).mean()

        # Total Weighting
        l_total = (
            (0_000.001 * l_budget) +
            (0_100.000 * l_gap) +
            (1_000.000 * l_efficiency) +
            (0_005.000 * l_sharp) +
            (0_050.000 * l_contrast)
        )

        return l_total, TokenizerLossStats(
            l_budget=l_budget.item(),
            l_gap=l_gap.item(),
            l_efficiency=l_efficiency.item(),
            l_sharp=l_sharp.item(),
            l_contrast=l_contrast.item(),
        )

In [177]:
tokenizer_loss = TokenizerLoss(tokenizer)

### Tokenizer - Training Loop

In [178]:
import pandas as pd

from IPython.display import display, clear_output

In [179]:
class TokernizerTrainerArguments(TypedDict):
    epochs: int
    learning_rate: float

    log_interval: int
    accelerator: str

In [180]:
class TokenizerTrainer():
    def __init__(
            self,
            model: TokenizerModel,
            loss_fn: TokenizerLoss,
            training_args: TokernizerTrainerArguments
        ):
        super().__init__()
        self.model = model
        self.loss_fn = loss_fn
        self.training_args = training_args

        self.optimizer = torch.optim.Adam(
            self.model.parameters(),
            lr=self.training_args['learning_rate']
        )

        self.history = pd.DataFrame(
            data=[],
            columns=[
                'epoch', 'loss',
                *TokenizerLossStats.__annotations__.keys()
            ]
        )

    def train(
            self,
            dataloader: DataLoader[Tuple[str, str]],
        ) -> None:
        device = torch.device(self.training_args['accelerator'])
        self.model.to(device)
        self.model.train()

        for epoch in range(self.training_args['epochs']):
            epoch_loss = 0.0
            epoch_stats = {k: 0.0 for k in TokenizerLossStats.__annotations__.keys()}

            for batch_idx, (input_seqs, _) in enumerate(dataloader):
                self.optimizer.zero_grad()

                input_tensors = stringBatchToTensor(input_seqs).to(device)
                y_pred = self.model(input_tensors)
                loss, stats = self.loss_fn(y_pred)

                loss.backward()
                self.optimizer.step()

                epoch_loss += loss.item()
                for k in epoch_stats.keys():
                    epoch_stats[k] += stats[k]

                print(
                    f"Epoch {epoch + 1}/{self.training_args['epochs']} "
                    f"Step {(batch_idx + 1)/len(dataloader)*100:.1f}% - ",
                    f"Loss: {epoch_loss/len(dataloader):.4f}",
                    end='\r'
                )

            avg_loss = epoch_loss / len(dataloader)
            avg_stats = {k: v / len(dataloader) for k, v in epoch_stats.items()}

            if (epoch + 1) % self.training_args['log_interval'] == 0:
                self.history = pd.concat([
                    self.history,
                    pd.DataFrame([{
                        'epoch': epoch + 1,
                        'loss': avg_loss,
                        **avg_stats
                    }])
                ], ignore_index=True)

                clear_output(wait=True)
                display(self.history)

In [181]:
tokenizer_trainer_args = TokernizerTrainerArguments(
    epochs=500,
    learning_rate=1e-2,
    log_interval=5,
    accelerator='cuda' if torch.cuda.is_available() else 'cpu'
)

tokenizer_trainer = TokenizerTrainer(
    model=tokenizer,
    loss_fn=tokenizer_loss,
    training_args=tokenizer_trainer_args
)

In [182]:
tokenizer_trainer.train(dataloader)

,epoch,loss,l_budget,l_gap,l_efficiency,l_sharp,l_contrast
0,5,59.725647,0.0,0.486896,0.002617,0.895597,0.078825
1,10,56.439010,0.0,0.451719,0.002671,0.887504,0.083164
2,15,56.957521,0.0,0.462756,0.002198,0.889656,0.080705
3,20,57.237602,0.0,0.460819,0.002615,0.887920,0.082029
4,25,55.941000,0.0,0.448572,0.002488,0.885957,0.083328
...,...,...,...,...,...,...,...
95,480,53.331900,0.0,0.422816,0.002477,0.884906,0.082969
96,485,54.793803,0.0,0.438123,0.002448,0.886285,0.082035
97,490,55.157398,0.0,0.443460,0.002290,0.887834,0.081641
98,495,54.486145,0.0,0.435785,0.002380,0.886431,0.081915


### Tokenizer - Inference

In [183]:
from IPython.display import display, HTML
import matplotlib
import matplotlib.cm as cm
import numpy as np

In [184]:
cmap = matplotlib.colormaps['berlin']

def inference(
        input_seq: list[str],
        y_pred: torch.Tensor,
        threshold: float = 0.8
    ):
    chars = input_seq[0]
    preds = y_pred[0].detach()
    activations = torch.sigmoid(8 * preds - 4).cpu().numpy()

    # Find subword boundaries
    boundaries = np.where(activations > threshold, 1, 0)
    subwords: List[Tuple[str, float, int]] = []  # (text, activation, start)
    current = ''
    start = 0
    for i, c in enumerate(chars):
        if boundaries[i]:
            if current:
                subwords.append((current, activations[i-1] if i > 0 else activations[i], start))
            current = c
            start = i
        else:
            current += c
    if current:
        subwords.append((current, activations[-1], start))

    main_html = ''
    for text, activation, start in subwords:
        rgb = tuple(int(255*x) for x in cmap(activation)[:3])

        subword_html = ''
        for j, c in enumerate(text):
            idx = start + j

            char_act: float = activations[idx]
            char_rgb = tuple(int(255*z) for z in cmap(char_act)[:3])

            c_esc = c.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
            c_esc = c_esc.replace(' ', '&nbsp;')
            c_esc = c_esc.replace('\n', '<br />')

            subword_html += f'<span style="border: 1px solid rgb{char_rgb};">{c_esc}</span>'
        main_html += f'<span class="subword" style="background-color: rgb{rgb};">{subword_html}</span>'

    style = (
        "font-family: monospace;"
        "font-size: 1.2em;"
    )
    subword_style = (
        "box-sizing: border-box;"
        "color: white;"
        "border: 1px solid black;"
    )
    char_style = (
        "box-sizing: border-box;"
    )
    html = f"""
    <div>
        <style>
            .subword {{{subword_style}}}
            .subword:hover {{
                filter: brightness(1.2);
                z-index: 1;
            }}
            .subword span {{{char_style}}}
        </style>
        <div style="{style}">{main_html}</div>
    </div>
    """
    display(HTML(html))

In [190]:
tokenizer.eval()

input_seq = next(iter(dataloader))[0]
device = next(tokenizer.parameters()).device

with torch.no_grad():
    y_pred = tokenizer(stringBatchToTensor(input_seq).to(device))

inference(
    input_seq=input_seq,
    y_pred=y_pred
)

### Tokenizer - Evaluation

In [186]:
from collections import Counter

def evaluate_tokenizer(
        tokenizer: TokenizerModel,
        dataloader: DataLoader[Tuple[str, str]],
        threshold: float = 0.8
    ) -> tuple[float, int, Counter[str]]:
    tokenizer.eval()
    device = next(tokenizer.parameters()).device
    total_tokens = 0
    total_docs = 0
    vocab_counter: Counter[str] = Counter()

    with torch.no_grad():
        for batch, _ in dataloader:
            input_tensors = stringBatchToTensor(batch).to(device)
            y_pred = tokenizer(input_tensors)
            for i, seq in enumerate(batch):
                chars = seq
                preds = y_pred[i].detach()
                activations = torch.sigmoid(8 * preds - 4).cpu().numpy()
                boundaries = np.where(activations > threshold, 1, 0)
                subwords: list[str] = []
                current = ''
                for j, c in enumerate(chars):
                    if boundaries[j]:
                        if current:
                            subwords.append(current)
                        current = c
                    else:
                        current += c
                if current:
                    subwords.append(current)
                total_tokens += len(subwords)
                vocab_counter.update(subwords)
                total_docs += 1

    avg_tokens_per_doc = total_tokens / total_docs if total_docs else 0
    vocab_size = len(vocab_counter)
    print(f"Average tokens per document: {avg_tokens_per_doc:.2f}")
    print(f"Vocabulary size (unique tokens): {vocab_size}")
    print("Top 10 subwords:")
    for subword, count in vocab_counter.most_common(10):
        print(f"  {repr(subword)}: {count}")
    return avg_tokens_per_doc, vocab_size, vocab_counter

In [187]:
avg_tokens_per_doc, vocab_size, vocab_counter = evaluate_tokenizer(tokenizer, dataloader)

Average tokens per document: 1112.16
Vocabulary size (unique tokens): 20820
Top 10 subwords:
  'f t': 527
  'he S': 517
  'R O': 397
  ' \n>': 371
  'ithub.c': 363
  'HE S': 356
  'ttps://g': 332
  'nd t': 317
  'ARRAN': 302
  'N A': 287


### Tokenizer - Export

In [ ]:
# :) No need to export useless models